In [ ]:
# 셀 1: 환경 설정 + import
import os
import sys
import sqlite3
from dotenv import load_dotenv

load_dotenv()

print(f"Python : {sys.version}")
print(f"OPENAI_API_KEY : {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정 ⚠️'}")
print(f"BYBIT_API_KEY  : {'설정됨' if os.getenv('BYBIT_API_KEY')  else '미설정 (샘플 모드)'}")

In [ ]:
# 셀 2: 그래프 + DEFAULT_STATE 임포트
from graph import graph, DEFAULT_STATE

print("graph       :", graph)
print("state keys  :", list(DEFAULT_STATE.keys()))

In [ ]:
# 셀 3: 그래프 시각화
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
# 셀 4: 한 사이클 실행
result = graph.invoke({**DEFAULT_STATE, "session_id": "demo"})
print("실행 완료")

In [ ]:
# 셀 5: 결과 출력
stats = result.get("stats", {})

print("=" * 60)
print("[KPI]")
print(f"  win_rate         : {stats.get('win_rate', 0):.1%}")
print(f"  avg_return_rate  : {stats.get('avg_return_rate', 0):.2f}%")
print(f"  expected_value   : {stats.get('expected_value', 0):.2f}%")
print(f"  loss_consistency : {stats.get('loss_consistency', 0):.2f}")

print("\n[약점 태그]")
weaknesses = result.get("weaknesses", [])
if weaknesses:
    for w in weaknesses:
        print(f"  • {w}")
else:
    print("  (없음)")

print("\n[action_rule]")
print(f"  {result.get('action_rule', '(없음)')}")

print("\n[코칭 결과]")
coaching = result.get("coaching_output", "")
print(coaching if coaching else "  (없음)")
print("=" * 60)

In [ ]:
# 셀 6: QA assert (D7-3)

# 환경 체크
assert graph is not None, "그래프 컴파일 실패"

# 저널 분석 플로우
result = graph.invoke({**DEFAULT_STATE, "session_id": "qa_test"})
assert "win_rate" in result.get("stats", {}), "win_rate 없음"
assert "avg_return_rate" in result.get("stats", {}), "avg_return_rate 없음"
assert "expected_value" in result.get("stats", {}), "expected_value 없음"
assert "loss_consistency" in result.get("stats", {}), "loss_consistency 없음"
assert result.get("action_rule", "") != "", "action_rule 비어있음"
print("✅ 저널 분석 PASS")

# 메모리 영속성
conn = sqlite3.connect("tradecoach.db")
rows = conn.execute(
    "SELECT avg_return_rate, expected_value, loss_consistency FROM trade_history ORDER BY id DESC LIMIT 1"
).fetchone()
assert rows is not None, "trade_history 저장 안됨"
conn.close()
print("✅ 메모리 영속성 PASS")

# 엣지 케이스
state_empty = {**DEFAULT_STATE, "last_fetched_at": "2099-01-01"}
result_empty = graph.invoke(state_empty)
assert result_empty.get("has_new_data") == False, "신규 없음 분기 실패"
print("✅ 엣지 케이스 PASS")

print("\n🎉 전체 QA 통과")